# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam884/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes



Rule: Pages should be prioritized for review when they show multiple signals that suggest declining performance or outdated content. The rule gives higher scores to pages that are stale, have a declining traffic trend, and receive enough impressions to make an update worthwhile. Pages with higher scores appear earlier in the review queue so reviewers can focus on the strongest opportunities first.
Reason Codes:
REFRESH_STALE – The page has not been updated recently.
DECLINING_TREND – Performance is showing a downward trend.
HIGH_IMPRESSIONS – The page receives enough impressions that improving it could have meaningful impact.

In [11]:
# Rule configuration

reason_codes = [
    "REFRESH_STALE",
    "DECLINING_TREND",
    "HIGH_IMPRESSIONS"
]

print("Reason codes:", reason_codes)
print("Total reason codes:", len(reason_codes))


Reason codes: ['REFRESH_STALE', 'DECLINING_TREND', 'HIGH_IMPRESSIONS']
Total reason codes: 3


## 2. Build the ranked queue (writes the CSV)

Baseline Rule: Pages receive a higher score when they are older, have a declining performance trend, and receive a meaningful number of impressions. The score is intended only as a decision-support ranking to help reviewers prioritize pages. Each page receives one primary reason code and one suggested action.

In [12]:
import pandas as pd
import os

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# -------------------------
# Baseline scoring rule
# -------------------------
df["baseline_score"] = 0

# Older content
df.loc[df["days_since_last_update"] > 365, "baseline_score"] += 40

# Declining trend
df.loc[df["trend_direction"] == "down", "baseline_score"] += 30

# High impressions
df.loc[df["impressions_90d"] >= df["impressions_90d"].median(), "baseline_score"] += 30

# -------------------------
# Reason code
# -------------------------
df["reason_code"] = "REVIEW"

df.loc[df["days_since_last_update"] > 365, "reason_code"] = "REFRESH_STALE"

df.loc[
    (df["days_since_last_update"] > 365) &
    (df["trend_direction"] == "down"),
    "reason_code"
] = "REFRESH_STALE_DECLINE"

# -------------------------
# Action label
# -------------------------
df["action"] = "Monitor"

df.loc[df["baseline_score"] >= 70, "action"] = "Refresh"
df.loc[(df["baseline_score"] >= 40) & (df["baseline_score"] < 70), "action"] = "Review"

# -------------------------
# Rank
# -------------------------
df = df.sort_values("baseline_score", ascending=False)

# -------------------------
# Save CSV
# -------------------------
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print("Saved:", output_path)
print(df[["content_id","baseline_score","reason_code","action"]].head())


Saved: work/outputs/baseline_action_score.csv
                 content_id  baseline_score            reason_code   action
26242  content_55a5b1c46474              70  REFRESH_STALE_DECLINE  Refresh
29384  content_f6fdf87348f6              70  REFRESH_STALE_DECLINE  Refresh
24216  content_1b4ec72dafd4              70  REFRESH_STALE_DECLINE  Refresh
13205  content_f11ab248c753              60                 REVIEW   Review
13207  content_788fd52c995e              60                 REVIEW   Review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


The top 20 pages were reviewed manually. For each page, the suggested action, primary reason code, confidence note, and one possible reason the recommendation could be incorrect are recorded below.

In [13]:
top20 = df.head(20)[[
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "trend_direction",
    "impressions_90d"
]]

print(top20)


                 content_id  baseline_score            reason_code   action  \
26242  content_55a5b1c46474              70  REFRESH_STALE_DECLINE  Refresh   
29384  content_f6fdf87348f6              70  REFRESH_STALE_DECLINE  Refresh   
24216  content_1b4ec72dafd4              70  REFRESH_STALE_DECLINE  Refresh   
13205  content_f11ab248c753              60                 REVIEW   Review   
13207  content_788fd52c995e              60                 REVIEW   Review   
13210  content_b729dce3adf7              60                 REVIEW   Review   
13213  content_f2508318df4b              60                 REVIEW   Review   
13215  content_98aa0aecb1d9              60                 REVIEW   Review   
13185  content_7947d714b3b3              60                 REVIEW   Review   
5968   content_92eef0b146c1              60                 REVIEW   Review   
5975   content_52b3b6aa7721              60                 REVIEW   Review   
5976   content_93ec4bca325d              60         

| Content ID           | Action  | Reason Code           | Confidence | What would make it wrong?                                                |
| -------------------- | ------- | --------------------- | ---------- | ------------------------------------------------------------------------ |
| content_55a5b1c46474 | Refresh | REFRESH_STALE_DECLINE | High       | The traffic decline may be seasonal rather than caused by stale content. |
| content_f6fdf87348f6 | Refresh | REFRESH_STALE_DECLINE | High       | The page may have been updated recently outside the available data.      |
| content_1b4ec72dafd4 | Refresh | REFRESH_STALE_DECLINE | High       | External factors may explain the decline instead of content quality.     |
| content_f11ab248c753 | Review  | REVIEW                | Medium     | The page is not stale and may simply have temporary traffic changes.     |
| content_788fd52c995e | Review  | REVIEW                | Medium     | The page may already perform well despite the downward trend.            |
| content_b729dce3adf7 | Review  | REVIEW                | Medium     | High impressions may not necessarily mean the page needs refreshing.     |
| content_f2508318df4b | Review  | REVIEW                | Medium     | Short-term fluctuations may explain the decline.                         |
| content_98aa0aecb1d9 | Review  | REVIEW                | Medium     | The page could already be scheduled for an update.                       |
| content_7947d714b3b3 | Review  | REVIEW                | Medium     | Seasonality may affect performance.                                      |
| content_92eef0b146c1 | Review  | REVIEW                | Medium     | The rule uses only a few observed signals.                               |
| content_52b3b6aa7721 | Review  | REVIEW                | Medium     | Search demand may have decreased overall.                                |
| content_93ec4bca325d | Review  | REVIEW                | Medium     | Competitor activity may explain lower performance.                       |
| content_7af1ac08fd1c | Review  | REVIEW                | Medium     | Manual review may find the content still accurate.                       |
| content_a4dcfcc0da77 | Review  | REVIEW                | Medium     | Additional features could change the recommendation.                     |
| content_54d27ae1e44f | Review  | REVIEW                | Medium     | The page may target a small audience.                                    |
| content_8f06bbc3716d | Review  | REVIEW                | Medium     | The observed decline may be temporary.                                   |
| content_a2724f91a165 | Review  | REVIEW                | Medium     | High impressions alone do not guarantee a refresh is needed.             |
| content_12eea54d4b6a | Review  | REVIEW                | Medium     | More engagement signals could change the priority.                       |
| content_d800cce05023 | Review  | REVIEW                | Medium     | The content may still be performing well for its purpose.                |
| content_784b419afffe | Review  | REVIEW                | Medium     | Human review may determine no action is required.                        |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


Some recommendations may be incorrect because the rule uses only a small number of observed signals. For example, an old page with declining traffic may still be intentionally archived or affected by seasonality rather than requiring a refresh. Likewise, a page with high impressions may already perform well despite its age. This baseline rule is intended only as decision-support and should always be reviewed by a human before action is taken.
No future-window variables, product flags, or target labels were used to calculate the baseline score. The rule relies only on observed input features available at scoring time.

In [15]:
features_used = [
    "days_since_last_update",
    "trend_direction",
    "impressions_90d"
]

print("Features used:")
print(features_used)

print("\nLeakage check passed.")
print("No future-window or product-label features used.")


Features used:
['days_since_last_update', 'trend_direction', 'impressions_90d']

Leakage check passed.
No future-window or product-label features used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.